In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from etl import GetData

In [3]:
# 1. Crear una instancia de la clase (esto proporciona el argumento 'self')
data_loader = GetData()

In [4]:
data_loader.download_data()

La carga se realizará directamente desde S3: s3://amico-udem/DataModels/costs.csv
Asegúrate de que tus credenciales de AWS estén configuradas (ej. variables de entorno AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, o ~/.aws/credentials).


True

In [5]:
df = data_loader.create_dataset()

Iniciando la lectura del dataset desde S3...
Lectura exitosa. DataFrame cargado con 352 filas.


In [6]:
if df is not None:
        print("\nDataFrame S3 cargado correctamente:")
        print(df.head())


DataFrame S3 cargado correctamente:
         Service  Relational Database Service($)  EC2-Instances($)  \
0  Service total                    34131.482733      23531.788153   
1     2024-06-01                       31.851525          4.612654   
2     2024-06-02                       61.039787               NaN   
3     2024-06-03                       71.416393         63.161250   
4     2024-06-04                       54.833184         68.402379   

        FSx($)  Elastic File System($)  EC2-Other($)  CloudWatch($)  \
0  5152.073356             2830.548352   2132.939335    1543.902136   
1    12.472440                1.789663      0.611443       0.036701   
2    12.460864                1.789663      0.322181       0.016653   
3    12.654853                1.790544      5.613880       0.429317   
4    12.658942                1.791199      5.682819       0.459109   

        S3($)  Elastic Load Balancing($)   Backup($)  \
0  778.604880                 758.963353  530.170434   
1  

In [7]:
from feature_engineer import FeatureEngineer

/Users/angeleduardogamarrarios/Repositorio_UDEM/MLops_AMICO/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
  # feature engineer
feature_engineer = FeatureEngineer(df)

In [9]:
df_engineered = feature_engineer.create_features()

In [10]:
dfboc=feature_engineer.create_features_day(df_engineered)

In [11]:
dfscalad=feature_engineer.create_features_etiquetado(dfboc)

In [12]:
dfscalad

,Relational Database Service($),EC2-Instances($),FSx($),Elastic File System($),EC2-Other($),CloudWatch($),S3($),Elastic Load Balancing($),Backup($),Key Management Service($),DataSync($),Secrets Manager($),Resilience Hub($),day_of_week,mahalanobis_distance,is_outlier_mahalanobis
date,,,,,,,,,,,,,,,,
2024-06-01,-0.592423,-0.999276,-1.060436,-0.976153,-0.904894,-0.495977,-1.318657,-0.283595,-0.984327,-0.798520,-0.203227,-0.549665,0.0,Saturday,1.676803,No
2024-06-02,-0.261585,-0.997331,-1.056020,-0.962626,-0.910742,-0.489094,-1.343367,-0.189216,-1.018993,-0.776971,-0.208098,-0.436052,0.0,Sunday,2.004497,No
2024-06-03,-0.332165,-0.707250,-1.141054,-0.964218,-0.641159,-0.537721,-1.399405,-0.506735,-1.017974,-0.644293,-0.158846,-0.730918,0.0,Monday,2.108012,No
2024-06-04,-0.552840,-0.594763,-1.144325,-0.978410,-0.754581,-0.528254,-1.403380,-0.645916,-1.013475,-0.707304,-0.143104,-0.912510,0.0,Tuesday,2.111300,No
2024-06-05,-0.570957,-0.516806,-1.178708,-0.970482,-0.709455,-0.568771,-1.423768,-0.688120,-1.022353,-0.613103,-0.262452,0.123839,0.0,Wednesday,2.248293,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-05-13,4.013976,0.424861,0.717311,1.877659,0.043480,1.402638,1.398555,3.683198,2.168519,3.004516,-0.143104,-0.237798,0.0,Tuesday,5.570766,Si
2025-05-14,3.994464,0.471894,0.738882,1.847187,0.123543,1.157719,1.398457,4.676995,2.109517,2.942045,-0.262452,-0.419929,0.0,Wednesday,5.587628,Si
2025-05-15,3.863563,0.483870,0.723159,1.870618,0.634860,1.388629,1.386483,5.264172,2.156304,2.968173,-0.323026,6.430990,0.0,Thursday,5.251031,Si


In [27]:
from Train import Train

In [28]:
TrainAmico=Train(dfscalad)

In [29]:
X_train, X_test, y_train, y_test=TrainAmico.train_test_split()

In [ ]:
!mlflow ui --port 5000

/Users/angeleduardogamarrarios/Repositorio_UDEM/MLops_AMICO/.venv/lib/python3.12/site-packages/mlflow/gateway/config.py:454: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Route(ConfigModel):
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [40510]
INFO:     Started server process [40515]
INFO:     Waiting for application startup.
INFO:     Started server process [40512]
INFO:     Waiting for application startup.
INFO:     Started server process [40513]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Applicat

In [43]:
from train_whit_mlflow_optuna import IsolationForestTrainer

In [44]:
TrainAmicoD=IsolationForestTrainer(dfscalad)

In [45]:
TrainAmicoD.preprocess_split()

In [48]:
TrainAmicoD._score_predictions(y_test,{'No': 0, 'Si': 1})

ValueError: Found input variables with inconsistent numbers of samples: [91, 2]